In [1]:

!pip install --no-index /kaggle/input/imc2024-packages-lightglue-rerun-kornia/kornia-0.7.2-py2.py3-none-any.whl --no-deps
!pip install --no-index /kaggle/input/imc2024-packages-lightglue-rerun-kornia/kornia_moons-0.2.9-py3-none-any.whl --no-deps
!pip install --no-index /kaggle/input/imc2024-packages-lightglue-rerun-kornia/lightglue-0.0-py3-none-any.whl --no-deps
!mkdir -p /root/.cache/torch/hub/checkpoints
!cp /kaggle/input/aliked/pytorch/aliked-n16/1/aliked-n16.pth /root/.cache/torch/hub/checkpoints/
!cp /kaggle/input/lightglue/pytorch/aliked/1/aliked_lightglue.pth /root/.cache/torch/hub/checkpoints/
!cp /kaggle/input/lightglue/pytorch/aliked/1/aliked_lightglue.pth /root/.cache/torch/hub/checkpoints/aliked_lightglue_v0-1_arxiv-pth


Processing /kaggle/input/imc2024-packages-lightglue-rerun-kornia/kornia-0.7.2-py2.py3-none-any.whl
  Attempting uninstall: kornia
    Found existing installation: kornia 0.8.1
    Uninstalling kornia-0.8.1:
      Successfully uninstalled kornia-0.8.1
Processing /kaggle/input/imc2024-packages-lightglue-rerun-kornia/kornia_moons-0.2.9-py3-none-any.whl
Processing /kaggle/input/imc2024-packages-lightglue-rerun-kornia/lightglue-0.0-py3-none-any.whl


In [2]:
import pandas as pd
import os
import sys
import shutil
import numpy as np
from scipy.spatial.transform import Rotation as R
import cv2
import torch
import pycolmap
import array
import h5py
import csv
import dataclasses
import subprocess
import kornia as K
import kornia.feature as KF
import torchvision.transforms as T
import torch.nn.functional as F
from lightglue import ALIKED
from lightglue import LightGlue, match_pair
from lightglue.utils import load_image
from kornia.io import ImageLoadType
from transformers import AutoImageProcessor, AutoModel
from itertools import combinations
sys.path.append('/kaggle/input/imc25-utils')
from database import COLMAPDatabase
from h5_to_db import add_keypoints, add_matches, import_into_colmap


/usr/local/lib/python3.11/dist-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.11/dist-packages/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
2025-06-01 16:56:46.490007: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748797006.699195      26 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748797006.762655      26 cuda_blas.cc:1418] Unable to register cuBLAS fac

In [3]:
MIN_PAIRS = 5  # Минимальное количество пар для изолированных изображений
TOLERANCE = 100  # Порог для исключения слишком далеких пар

def get_image_pairs(dataset, scene, group, device='cuda' if torch.cuda.is_available() else 'cpu'):
    if not os.path.exists(dataset):
        return []
    
    image_paths = [os.path.join(dataset, row['image']) for idx, row in group]
    indices = [idx for idx, row in group]

    if len(image_paths) < 2:
        return []
    if len(image_paths) <= 3:
        return list(combinations(indices, 2))
    
    processor = AutoImageProcessor.from_pretrained('/kaggle/input/dinov2/pytorch/base/1/')
    model = AutoModel.from_pretrained('/kaggle/input/dinov2/pytorch/base/1/').eval().to(device)

    embeddings = []
    for img_path in image_paths:
        image = K.io.load_image(img_path, K.io.ImageLoadType.RGB32, device=device)[None, ...]
        with torch.inference_mode():
            inputs = processor(images=image, return_tensors="pt", do_rescale=False, 
                             do_resize=True, do_center_crop=True, size=224).to(device)
            outputs = model(**inputs)
            embedding = F.normalize(outputs.last_hidden_state.max(dim=1)[0])
        embeddings.append(embedding)
    
    embeddings = torch.cat(embeddings, dim=0)
    distances = torch.cdist(embeddings, embeddings).cpu().numpy()
    distances_ = distances <= 0.3
    np.fill_diagonal(distances_, False)
    z = distances_.sum(axis=1)
    idxs0 = np.where(z == 0)[0]
    for idx0 in idxs0:
        t = np.argsort(distances[idx0])[1:MIN_PAIRS + 1]
        distances_[idx0, t] = True

    s = np.where(distances >= TOLERANCE)
    distances_[s] = False
    pairs = []
    for i in range(len(image_paths)):
        for j in range(len(image_paths)):
            if distances_[i][j]:
                if i < j:
                    pairs.append((indices[i], indices[j]))
                else:
                    pairs.append((indices[j], indices[i]))
    
    pairs = list(set(pairs))
    return pairs

In [4]:
data_path = "/kaggle/input/image-matching-challenge-2025"
test_path = os.path.join(data_path, "test")
train_path = os.path.join(data_path, "train")

submission = pd.read_csv(os.path.join(data_path, "sample_submission.csv"))
@dataclasses.dataclass
class Prediction:
    image_id: str
    dataset: str
    scene: str
    filename: str
    cluster_index: int = None
    rotation: np.ndarray = None
    translation: np.ndarray = None

work_dir = '/kaggle/working/result'
os.makedirs(work_dir, exist_ok=True)

submission_csv = os.path.join(data_path, 'sample_submission.csv')
# Чтение CSV и создание списка samples
with open(submission_csv, 'r') as file:
    reader = csv.DictReader(file)
    rows = list(reader)

samples = [Prediction(
    image_id=row['image_id'],
    dataset=row['dataset'],
    scene=row['scene'],
    filename=row['image']
) for row in rows]

print('Datasets loaded:')
datasets = set(row['dataset'] for row in rows)
for dataset in datasets:
    count = sum(1 for row in rows if row['dataset'] == dataset)
    print(f'  - {dataset}: {count} images')

Datasets loaded:
  - pt_stpeters_stpauls: 200 images
  - ETs: 22 images
  - imc2023_haiper: 54 images
  - stairs: 51 images
  - imc2024_dioscuri_baalshamin: 138 images
  - imc2024_lizard_pond: 214 images
  - fbk_vineyard: 163 images
  - pt_brandenburg_british_buckingham: 225 images
  - imc2023_heritage: 209 images
  - pt_sacrecoeur_trevi_tajmahal: 225 images
  - pt_piazzasanmarco_grandplace: 168 images
  - amy_gardens: 200 images
  - imc2023_theather_imc2024_church: 76 images


In [5]:
def process_group(dataset, scene, group, indices, pairs, samples):
    project_dir = f"/kaggle/working/project_{dataset}_{scene}_{indices[0]}"
    os.makedirs(project_dir, exist_ok=True)
    db_path = os.path.join(project_dir, "database.db")
    
    if os.path.exists(db_path):
        os.remove(db_path)

    test_path = "/kaggle/input/image-matching-challenge-2025/test"
    existing_image_names = []
    non_existing_indices = []
    row_idx_to_existing_idx = {}
    for position, (idx, row) in enumerate(group):
        image_path = os.path.join(test_path, dataset, row['image'])
        if os.path.exists(image_path):
            image_name = os.path.basename(row['image'])
            dest_path = os.path.join(project_dir, image_name)
            shutil.copy(image_path, dest_path)
            existing_image_names.append(image_name)
            row_idx_to_existing_idx[idx] = len(existing_image_names) - 1
        else:
            non_existing_indices.append(idx)
    
    # Обработка несуществующих изображений как outliers
    for row_idx in non_existing_indices:
        samples[row_idx].cluster_index = None
        samples[row_idx].rotation = None
        samples[row_idx].translation = None
    
    if len(existing_image_names) < 2:
        # Если существующих изображений меньше 2, все считаются outliers
        for idx, _ in group:
            samples[idx].cluster_index = None
            samples[idx].rotation = None
            samples[idx].translation = None
        shutil.rmtree(project_dir)
        return
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    aliked = ALIKED(max_num_keypoints=(4096*4), detection_threshold=0.3, resize=512).eval().to(device)
    matcher = KF.LightGlueMatcher(
        "aliked",
        {
            "width_confidence": 0.9,
            "depth_confidence": 0.9,
            "mp": True if 'cuda' in str(device) else False
        }
    ).eval().to(device)
    
    feature_dir = os.path.join(project_dir, "features")
    os.makedirs(feature_dir, exist_ok=True)

    for h5_file in ['keypoints.h5', 'descriptors.h5', 'matches.h5']:
        h5_path = os.path.join(feature_dir, h5_file)
        if os.path.exists(h5_path):
            os.remove(h5_path)
            
    with h5py.File(os.path.join(feature_dir, 'keypoints.h5'), 'w') as f_kp, \
         h5py.File(os.path.join(feature_dir, 'descriptors.h5'), 'w') as f_desc:
        for image_name in existing_image_names:
            with torch.inference_mode():
                img = load_image(os.path.join(project_dir, image_name)).to(device)
                feats = aliked.extract(img)
                kpts = feats['keypoints'].squeeze().cpu().numpy()
                descs = feats['descriptors'].squeeze().detach().cpu().numpy()
                f_kp[image_name] = kpts
                f_desc[image_name] = descs

    keypoints = {}
    descriptors = {}
    with h5py.File(os.path.join(feature_dir, 'keypoints.h5'), 'r') as f_kp, \
         h5py.File(os.path.join(feature_dir, 'descriptors.h5'), 'r') as f_desc:
        for image_name in existing_image_names:
            keypoints[image_name] = f_kp[image_name][...]
            descriptors[image_name] = f_desc[image_name][...]
            
    MIN_MATCHES = 125
    with h5py.File(os.path.join(feature_dir, 'matches.h5'), 'w') as f_match:
        for idx1, idx2 in pairs:
            if idx1 in row_idx_to_existing_idx and idx2 in row_idx_to_existing_idx:
                image_name1 = existing_image_names[row_idx_to_existing_idx[idx1]]
                image_name2 = existing_image_names[row_idx_to_existing_idx[idx2]]
                kp1 = torch.from_numpy(keypoints[image_name1]).to(device)
                kp2 = torch.from_numpy(keypoints[image_name2]).to(device)
                desc1 = torch.from_numpy(descriptors[image_name1]).to(device)
                desc2 = torch.from_numpy(descriptors[image_name2]).to(device)
                laf1 = KF.laf_from_center_scale_ori(kp1[None])
                laf2 = KF.laf_from_center_scale_ori(kp2[None])
                with torch.inference_mode():
                    dists, idxs = matcher(desc1, desc2, laf1, laf2)
                if len(idxs) >= MIN_MATCHES:
                    matches = idxs.cpu().numpy().astype(np.uint32)
                    group = f_match.require_group(image_name1)
                    group.create_dataset(image_name2, data=matches)

    db_name = db_path
    db = COLMAPDatabase.connect(db_name)
    db.create_tables()
    fname_to_id = add_keypoints(db, feature_dir, project_dir, '', 'simple-pinhole', False)
    add_matches(db, feature_dir, fname_to_id)
    db.commit()

    pycolmap.match_exhaustive(db_name, sift_options={'num_threads':1})
    maps = pycolmap.incremental_mapping(
        database_path=db_path,
        image_path=project_dir,
        output_path='/kaggle/working/',
        options=pycolmap.IncrementalPipelineOptions({'min_model_size':5, 'max_num_models':3, 'num_threads':1})
    )
    
    reconstructed_images = set()
    if maps:
        for map_index, cur_map in maps.items():
            for image_id, image in cur_map.images.items():
                image_name = image.name
                if image_name in existing_image_names:
                    existing_idx = existing_image_names.index(image_name)
                    for row_idx, e_idx in row_idx_to_existing_idx.items():
                        if e_idx == existing_idx:
                            samples[row_idx].cluster_index = map_index
                            samples[row_idx].rotation = image.cam_from_world.rotation.matrix()
                            samples[row_idx].translation = image.cam_from_world.translation
                            reconstructed_images.add(image_name)
                            break
    print(existing_image_names)
    print(non_existing_indices)
    # Сброс значений для нереконструированных существующих изображений
    for image_name in existing_image_names:
        if image_name not in reconstructed_images:
            existing_idx = existing_image_names.index(image_name)
            for row_idx, e_idx in row_idx_to_existing_idx.items():
                if e_idx == existing_idx:
                    samples[row_idx].cluster_index = None
                    samples[row_idx].rotation = None
                    samples[row_idx].translation = None
                    break
    
    # Повторное подтверждение outliers для несуществующих изображений
    for row_idx in non_existing_indices:
        samples[row_idx].cluster_index = None
        samples[row_idx].rotation = None
        samples[row_idx].translation = None
    
    shutil.rmtree(project_dir)
    
    

In [6]:
def write_submission(samples):
    array_to_str = lambda array: ';'.join([f"{x:.09f}" for x in array]) if array is not None else ';'.join(['nan'] * 9)
    none_to_str = lambda n: ';'.join(['nan'] * n)
    submission_file = '/kaggle/working/submission.csv'
    with open(submission_file, 'w') as f:
        header = 'image_id,dataset,scene,image,rotation_matrix,translation_vector\n'
        f.write(header)
        for pred in samples:
            cluster = 'outliers' if pred.cluster_index is None else f'cluster{pred.cluster_index}'
            rot = none_to_str(9) if pred.rotation is None else array_to_str(pred.rotation.flatten())
            trans = none_to_str(3) if pred.translation is None else array_to_str(pred.translation)
            f.write(f'{pred.image_id},{pred.dataset},{cluster},{pred.filename},{rot},{trans}\n')
    
    print(f'📄 Submission file created: {submission_file}')
    !head {submission_file}

In [7]:
# Группировка данных
groups = {}
for idx, row in enumerate(rows):
    key = (row['dataset'], row['scene'])
    if key not in groups:
        groups[key] = []
    groups[key].append((idx, row))

os.environ['QT_QPA_PLATFORM'] = 'offscreen'

for (dataset, scene), group in groups.items():
    print(dataset)
    dataset_path = os.path.join(test_path, dataset)
    pairs = get_image_pairs(dataset_path, scene, group)
    if pairs:
        process_group(dataset, scene, group, [idx for idx, _ in group], pairs, samples)

write_submission(samples)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


ETs
Loaded LightGlue model


 36%|███▌      | 38/105 [00:00<00:00, 4229.81it/s]
I20250601 16:57:11.597978 140342585136704 misc.cc:44] 
Feature matching
I20250601 16:57:11.598608 140342576744000 sift.cc:1432] Creating SIFT CPU feature matcher
I20250601 16:57:11.598769 140342585136704 pairing.cc:168] Generating exhaustive image pairs...
I20250601 16:57:11.598796 140342585136704 pairing.cc:201] Matching block [1/1, 1/1]
I20250601 16:57:11.909714 140342585136704 feature_matching.cc:46] in 0.311s
I20250601 16:57:11.910464 140342585136704 timer.cc:91] Elapsed time: 0.005 [minutes]
I20250601 16:57:11.913639 140350569600128 incremental_pipeline.cc:237] Loading database
I20250601 16:57:11.914494 140350569600128 database_cache.cc:66] Loading cameras...
I20250601 16:57:11.914561 140350569600128 database_cache.cc:76]  22 in 0.000s
I20250601 16:57:11.914576 140350569600128 database_cache.cc:84] Loading matches...
I20250601 16:57:11.914838 140350569600128 database_cache.cc:89]  38 in 0.000s
I20250601 16:57:11.914854 14035056960

['another_et_another_et001.png', 'another_et_another_et002.png', 'another_et_another_et003.png', 'another_et_another_et004.png', 'another_et_another_et005.png', 'another_et_another_et006.png', 'another_et_another_et007.png', 'another_et_another_et008.png', 'another_et_another_et009.png', 'another_et_another_et010.png', 'et_et000.png', 'et_et001.png', 'et_et002.png', 'et_et003.png', 'et_et004.png', 'et_et005.png', 'et_et006.png', 'et_et007.png', 'et_et008.png', 'outliers_out_et001.png', 'outliers_out_et002.png', 'outliers_out_et003.png']
[]
amy_gardens
fbk_vineyard
imc2023_haiper
imc2023_heritage
imc2023_theather_imc2024_church
imc2024_dioscuri_baalshamin
imc2024_lizard_pond
pt_brandenburg_british_buckingham
pt_piazzasanmarco_grandplace
pt_sacrecoeur_trevi_tajmahal
pt_stpeters_stpauls
stairs
Loaded LightGlue model


 52%|█████▏    | 11/21 [00:00<00:00, 2587.19it/s]
I20250601 16:57:40.200597 140342568351296 misc.cc:44] 
Feature matching
I20250601 16:57:40.200901 140342576744000 sift.cc:1432] Creating SIFT CPU feature matcher
I20250601 16:57:40.201099 140342568351296 pairing.cc:168] Generating exhaustive image pairs...
I20250601 16:57:40.201123 140342568351296 pairing.cc:201] Matching block [1/2, 1/2]
I20250601 16:57:40.609641 140342568351296 feature_matching.cc:46] in 0.409s
I20250601 16:57:40.609873 140342568351296 pairing.cc:201] Matching block [1/2, 2/2]
I20250601 16:57:40.609894 140342568351296 feature_matching.cc:46] in 0.000s
I20250601 16:57:40.609905 140342568351296 pairing.cc:201] Matching block [2/2, 1/2]
I20250601 16:57:40.610571 140342568351296 feature_matching.cc:46] in 0.001s
I20250601 16:57:40.610834 140342568351296 pairing.cc:201] Matching block [2/2, 2/2]
I20250601 16:57:40.610852 140342568351296 feature_matching.cc:46] in 0.000s
I20250601 16:57:40.610861 140342568351296 timer.cc:91

['stairs_split_1_1710453576271.png', 'stairs_split_1_1710453601885.png', 'stairs_split_1_1710453606287.png', 'stairs_split_1_1710453612890.png', 'stairs_split_1_1710453616892.png', 'stairs_split_1_1710453620694.png', 'stairs_split_1_1710453626698.png', 'stairs_split_1_1710453643106.png', 'stairs_split_1_1710453651110.png', 'stairs_split_1_1710453659313.png', 'stairs_split_1_1710453663515.png', 'stairs_split_1_1710453667117.png', 'stairs_split_1_1710453668718.png', 'stairs_split_1_1710453675921.png', 'stairs_split_1_1710453678922.png', 'stairs_split_1_1710453683725.png', 'stairs_split_1_1710453689727.png', 'stairs_split_1_1710453693529.png', 'stairs_split_1_1710453697531.png', 'stairs_split_1_1710453704934.png', 'stairs_split_1_1710453901046.png', 'stairs_split_1_1710453912451.png', 'stairs_split_1_1710453930259.png', 'stairs_split_1_1710453947066.png', 'stairs_split_1_1710453955270.png', 'stairs_split_1_1710453963274.png', 'stairs_split_1_1710453985484.png', 'stairs_split_1_17104539902

I20250601 16:57:41.012046 140350569600128 incremental_pipeline.cc:286] => No good initial image pair found.
I20250601 16:57:41.012147 140350569600128 incremental_pipeline.cc:282] Finding good initial image pair
I20250601 16:57:41.012376 140350569600128 incremental_pipeline.cc:286] => No good initial image pair found.
I20250601 16:57:41.012485 140350569600128 incremental_pipeline.cc:282] Finding good initial image pair
I20250601 16:57:41.012846 140350569600128 incremental_pipeline.cc:286] => No good initial image pair found.
I20250601 16:57:41.013010 140350569600128 incremental_pipeline.cc:282] Finding good initial image pair
I20250601 16:57:41.013270 140350569600128 incremental_pipeline.cc:286] => No good initial image pair found.
I20250601 16:57:41.013357 140350569600128 incremental_pipeline.cc:282] Finding good initial image pair
I20250601 16:57:41.013632 140350569600128 incremental_pipeline.cc:286] => No good initial image pair found.
I20250601 16:57:41.013798 140350569600128 increm